# Banking Transactions Analytics

In [1]:

!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder.appName("BankingAnalytics").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

customers_csv = """customer_id,name,city,age,account_type,signup_date
1,Rahul,Hyderabad,29,Savings,2023-01-10
2,Sneha,Bangalore,32,Current,2023-02-12
3,Arjun,Mumbai,27,Savings,2023-03-14
4,Priya,Delhi,35,Savings,2023-04-15
5,Karan,Chennai,30,Current,2023-05-11
6,Meera,Hyderabad,31,Savings,2023-06-10
7,Amit,Pune,38,Current,2023-06-22
8,Neha,Delhi,26,Savings,2023-07-10
9,Divya,Bangalore,28,Savings,2023-07-15
10,Vikram,Mumbai,42,Current,2023-08-01
11,Farhan,Hyderabad,34,Savings,2023-08-10
12,Simran,Delhi,25,Savings,2023-08-21
"""

accounts_csv = """account_id,customer_id,branch,balance
1001,1,Hyderabad Main,85000
1002,2,Bangalore Central,120000
1003,3,Mumbai West,45000
1004,4,Delhi North,95000
1005,5,Chennai South,60000
1006,6,Hyderabad Main,150000
1007,7,Pune East,30000
1008,8,Delhi North,70000
1009,9,Bangalore Central,110000
1010,10,Mumbai West,200000
1011,11,Hyderabad Main,50000
1012,12,Delhi North,40000
"""

transactions_csv = """txn_id,account_id,txn_type,amount,txn_date
1,1001,Credit,25000,2024-03-01
2,1002,Debit,15000,2024-03-01
3,1003,Credit,10000,2024-03-02
4,1004,Debit,5000,2024-03-02
5,1005,Credit,30000,2024-03-03
6,1006,Debit,20000,2024-03-03
7,1007,Credit,8000,2024-03-04
8,1008,Debit,12000,2024-03-04
9,1009,Credit,40000,2024-03-05
10,1010,Debit,35000,2024-03-05
11,1001,Debit,7000,2024-03-06
12,1002,Credit,18000,2024-03-06
13,1006,Credit,50000,2024-03-07
14,1010,Credit,60000,2024-03-07
15,1011,Debit,9000,2024-03-08
16,1012,Credit,16000,2024-03-08
17,1003,Debit,4000,2024-03-09
18,1004,Credit,22000,2024-03-09
19,1005,Debit,11000,2024-03-10
20,1009,Debit,14000,2024-03-10
"""

branches_csv = """branch,region,manager
Hyderabad Main,South,Ramesh
Bangalore Central,South,Leena
Mumbai West,West,Joseph
Delhi North,North,Sara
Chennai South,South,Kumar
Pune East,West,Anita
"""

logs_txt = """Rahul login
Sneha login
Rahul transfer
Arjun login
Priya withdrawal
Rahul logout
Meera login
Vikram transfer
Divya login
Farhan login
Simran withdrawal
Neha login
Amit deposit
Karan login
Meera transfer
Vikram login
Rahul deposit
Sneha withdrawal
Farhan transfer
Divya logout
"""

customer_profiles_json = """[
  {"customer_id": 1, "name": "Rahul", "contact": {"email": "rahul@mail.com", "phone": "9000011111"}, "services": ["UPI", "Credit Card", "Net Banking"]},
  {"customer_id": 2, "name": "Sneha", "contact": {"email": "sneha@mail.com", "phone": "9000022222"}, "services": ["UPI", "Debit Card"]},
  {"customer_id": 3, "name": "Arjun", "contact": {"email": "arjun@mail.com", "phone": "9000033333"}, "services": ["Net Banking", "Loan"]},
  {"customer_id": 6, "name": "Meera", "contact": {"email": "meera@mail.com", "phone": "9000066666"}, "services": ["UPI", "Credit Card", "Loan"]},
  {"customer_id": 10, "name": "Vikram", "contact": {"email": "vikram@mail.com", "phone": "9000101010"}, "services": ["Net Banking", "Wealth"]}
]
"""

with open("bank_customers.csv", "w") as f: f.write(customers_csv)
with open("accounts.csv", "w") as f: f.write(accounts_csv)
with open("transactions.csv", "w") as f: f.write(transactions_csv)
with open("branches.csv", "w") as f: f.write(branches_csv)
with open("bank_logs.txt", "w") as f: f.write(logs_txt)
with open("customer_profiles.json", "w") as f: f.write(customer_profiles_json)

customers     = spark.read.csv("bank_customers.csv", header=True, inferSchema=True)
accounts      = spark.read.csv("accounts.csv",       header=True, inferSchema=True)
transactions  = spark.read.csv("transactions.csv",   header=True, inferSchema=True)
branches      = spark.read.csv("branches.csv",       header=True, inferSchema=True)
profiles      = spark.read.json("customer_profiles.json", multiLine=True)

print("Banking datasets created and loaded successfully")

Banking datasets created and loaded successfully


## Section 1 — DataFrame Basics

In [2]:
# 1. Show all customers
customers.show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [3]:
# 2. Show all accounts
accounts.show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1001|          1|   Hyderabad Main|  85000|
|      1002|          2|Bangalore Central| 120000|
|      1003|          3|      Mumbai West|  45000|
|      1004|          4|      Delhi North|  95000|
|      1005|          5|    Chennai South|  60000|
|      1006|          6|   Hyderabad Main| 150000|
|      1007|          7|        Pune East|  30000|
|      1008|          8|      Delhi North|  70000|
|      1009|          9|Bangalore Central| 110000|
|      1010|         10|      Mumbai West| 200000|
|      1011|         11|   Hyderabad Main|  50000|
|      1012|         12|      Delhi North|  40000|
+----------+-----------+-----------------+-------+



In [4]:
# 3. Show all transactions
transactions.show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    11|      1001|   Debit|  7000|2024-03-06|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    15|      1011|   Debit|  9000|2024-03-08|
|    16|      1012|  Credit| 16000|2024-03-08|
|    17|      1003|   Debit|  4000|2024-03-09|
|    18|      1004|  Credit| 22000|2024-03-09|
|    19|     

In [5]:
# 4. Show all branches
branches.show()

+-----------------+------+-------+
|           branch|region|manager|
+-----------------+------+-------+
|   Hyderabad Main| South| Ramesh|
|Bangalore Central| South|  Leena|
|      Mumbai West|  West| Joseph|
|      Delhi North| North|   Sara|
|    Chennai South| South|  Kumar|
|        Pune East|  West|  Anita|
+-----------------+------+-------+



In [6]:
# 5. Print schema of customers
customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- account_type: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [7]:
# 6. Print schema of transactions
transactions.printSchema()

root
 |-- txn_id: integer (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- txn_date: date (nullable = true)



In [8]:
# 7. Count total customers
print("Total customers:", customers.count())

Total customers: 12


In [9]:
# 8. Count total accounts
print("Total accounts:", accounts.count())

Total accounts: 12


In [10]:
# 9. Count total transactions
print("Total transactions:", transactions.count())

Total transactions: 20


In [11]:
# 10. Display first 5 transactions
transactions.show(5)

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
+------+----------+--------+------+----------+
only showing top 5 rows


## Section 2 — Select, Rename, Filter

In [12]:
# 11. Select customer name, city, and account type
customers.select("name", "city", "account_type").show()

+------+---------+------------+
|  name|     city|account_type|
+------+---------+------------+
| Rahul|Hyderabad|     Savings|
| Sneha|Bangalore|     Current|
| Arjun|   Mumbai|     Savings|
| Priya|    Delhi|     Savings|
| Karan|  Chennai|     Current|
| Meera|Hyderabad|     Savings|
|  Amit|     Pune|     Current|
|  Neha|    Delhi|     Savings|
| Divya|Bangalore|     Savings|
|Vikram|   Mumbai|     Current|
|Farhan|Hyderabad|     Savings|
|Simran|    Delhi|     Savings|
+------+---------+------------+



In [13]:
# 12. Select account_id and balance
accounts.select("account_id", "balance").show()

+----------+-------+
|account_id|balance|
+----------+-------+
|      1001|  85000|
|      1002| 120000|
|      1003|  45000|
|      1004|  95000|
|      1005|  60000|
|      1006| 150000|
|      1007|  30000|
|      1008|  70000|
|      1009| 110000|
|      1010| 200000|
|      1011|  50000|
|      1012|  40000|
+----------+-------+



In [14]:
# 13. Rename balance to current_balance
accounts.withColumnRenamed("balance", "current_balance").show()

+----------+-----------+-----------------+---------------+
|account_id|customer_id|           branch|current_balance|
+----------+-----------+-----------------+---------------+
|      1001|          1|   Hyderabad Main|          85000|
|      1002|          2|Bangalore Central|         120000|
|      1003|          3|      Mumbai West|          45000|
|      1004|          4|      Delhi North|          95000|
|      1005|          5|    Chennai South|          60000|
|      1006|          6|   Hyderabad Main|         150000|
|      1007|          7|        Pune East|          30000|
|      1008|          8|      Delhi North|          70000|
|      1009|          9|Bangalore Central|         110000|
|      1010|         10|      Mumbai West|         200000|
|      1011|         11|   Hyderabad Main|          50000|
|      1012|         12|      Delhi North|          40000|
+----------+-----------+-----------------+---------------+



In [15]:
# 14. Rename txn_type to transaction_type
transactions.withColumnRenamed("txn_type", "transaction_type").show()

+------+----------+----------------+------+----------+
|txn_id|account_id|transaction_type|amount|  txn_date|
+------+----------+----------------+------+----------+
|     1|      1001|          Credit| 25000|2024-03-01|
|     2|      1002|           Debit| 15000|2024-03-01|
|     3|      1003|          Credit| 10000|2024-03-02|
|     4|      1004|           Debit|  5000|2024-03-02|
|     5|      1005|          Credit| 30000|2024-03-03|
|     6|      1006|           Debit| 20000|2024-03-03|
|     7|      1007|          Credit|  8000|2024-03-04|
|     8|      1008|           Debit| 12000|2024-03-04|
|     9|      1009|          Credit| 40000|2024-03-05|
|    10|      1010|           Debit| 35000|2024-03-05|
|    11|      1001|           Debit|  7000|2024-03-06|
|    12|      1002|          Credit| 18000|2024-03-06|
|    13|      1006|          Credit| 50000|2024-03-07|
|    14|      1010|          Credit| 60000|2024-03-07|
|    15|      1011|           Debit|  9000|2024-03-08|
|    16|  

In [16]:
# 15. Show customers from Hyderabad
customers.filter(F.col("city") == "Hyderabad").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
+-----------+------+---------+---+------------+-----------+



In [17]:
# 16. Show customers older than 30
customers.filter(F.col("age") > 30).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
+-----------+------+---------+---+------------+-----------+



In [18]:
# 17. Show Savings account customers
customers.filter(F.col("account_type") == "Savings").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [19]:
# 18. Show accounts with balance greater than 100000
accounts.filter(F.col("balance") > 100000).show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1002|          2|Bangalore Central| 120000|
|      1006|          6|   Hyderabad Main| 150000|
|      1009|          9|Bangalore Central| 110000|
|      1010|         10|      Mumbai West| 200000|
+----------+-----------+-----------------+-------+



In [20]:
# 19. Show Credit transactions
transactions.filter(F.col("txn_type") == "Credit").show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    16|      1012|  Credit| 16000|2024-03-08|
|    18|      1004|  Credit| 22000|2024-03-09|
+------+----------+--------+------+----------+



In [21]:
# 20. Show transactions with amount greater than 20000
transactions.filter(F.col("amount") > 20000).show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     5|      1005|  Credit| 30000|2024-03-03|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    18|      1004|  Credit| 22000|2024-03-09|
+------+----------+--------+------+----------+



## Section 3 — Sorting and Limit

In [22]:
# 21. Sort customers by age ascending
customers.orderBy(F.col("age").asc()).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
+-----------+------+---------+---+------------+-----------+



In [23]:
# 22. Sort customers by age descending
customers.orderBy(F.col("age").desc()).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [24]:
# 23. Sort accounts by balance descending
accounts.orderBy(F.col("balance").desc()).show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1010|         10|      Mumbai West| 200000|
|      1006|          6|   Hyderabad Main| 150000|
|      1002|          2|Bangalore Central| 120000|
|      1009|          9|Bangalore Central| 110000|
|      1004|          4|      Delhi North|  95000|
|      1001|          1|   Hyderabad Main|  85000|
|      1008|          8|      Delhi North|  70000|
|      1005|          5|    Chennai South|  60000|
|      1011|         11|   Hyderabad Main|  50000|
|      1003|          3|      Mumbai West|  45000|
|      1012|         12|      Delhi North|  40000|
|      1007|          7|        Pune East|  30000|
+----------+-----------+-----------------+-------+



In [25]:
# 24. Show top 5 highest balance accounts
accounts.orderBy(F.col("balance").desc()).show(5)

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1010|         10|      Mumbai West| 200000|
|      1006|          6|   Hyderabad Main| 150000|
|      1002|          2|Bangalore Central| 120000|
|      1009|          9|Bangalore Central| 110000|
|      1004|          4|      Delhi North|  95000|
+----------+-----------+-----------------+-------+
only showing top 5 rows


In [26]:
# 25. Show bottom 5 lowest balance accounts
accounts.orderBy(F.col("balance").asc()).show(5)

+----------+-----------+--------------+-------+
|account_id|customer_id|        branch|balance|
+----------+-----------+--------------+-------+
|      1007|          7|     Pune East|  30000|
|      1012|         12|   Delhi North|  40000|
|      1003|          3|   Mumbai West|  45000|
|      1011|         11|Hyderabad Main|  50000|
|      1005|          5| Chennai South|  60000|
+----------+-----------+--------------+-------+
only showing top 5 rows


In [27]:
# 26. Sort transactions by amount descending
transactions.orderBy(F.col("amount").desc()).show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|    14|      1010|  Credit| 60000|2024-03-07|
|    13|      1006|  Credit| 50000|2024-03-07|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|     5|      1005|  Credit| 30000|2024-03-03|
|     1|      1001|  Credit| 25000|2024-03-01|
|    18|      1004|  Credit| 22000|2024-03-09|
|     6|      1006|   Debit| 20000|2024-03-03|
|    12|      1002|  Credit| 18000|2024-03-06|
|    16|      1012|  Credit| 16000|2024-03-08|
|     2|      1002|   Debit| 15000|2024-03-01|
|    20|      1009|   Debit| 14000|2024-03-10|
|     8|      1008|   Debit| 12000|2024-03-04|
|    19|      1005|   Debit| 11000|2024-03-10|
|     3|      1003|  Credit| 10000|2024-03-02|
|    15|      1011|   Debit|  9000|2024-03-08|
|     7|      1007|  Credit|  8000|2024-03-04|
|    11|      1001|   Debit|  7000|2024-03-06|
|     4|     

In [28]:
# 27. Show top 3 biggest transactions
transactions.orderBy(F.col("amount").desc()).show(3)

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|    14|      1010|  Credit| 60000|2024-03-07|
|    13|      1006|  Credit| 50000|2024-03-07|
|     9|      1009|  Credit| 40000|2024-03-05|
+------+----------+--------+------+----------+
only showing top 3 rows


In [29]:
# 28. Sort transactions by txn_date
transactions.orderBy(F.col("txn_date").asc()).show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    11|      1001|   Debit|  7000|2024-03-06|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    15|      1011|   Debit|  9000|2024-03-08|
|    16|      1012|  Credit| 16000|2024-03-08|
|    17|      1003|   Debit|  4000|2024-03-09|
|    18|      1004|  Credit| 22000|2024-03-09|
|    19|     

In [30]:
# 29. Sort customers by city and age
customers.orderBy(F.col("city").asc(), F.col("age").asc()).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
+-----------+------+---------+---+------------+-----------+



In [31]:
# 30. Show latest 5 transactions
transactions.orderBy(F.col("txn_date").desc()).show(5)

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|    19|      1005|   Debit| 11000|2024-03-10|
|    20|      1009|   Debit| 14000|2024-03-10|
|    18|      1004|  Credit| 22000|2024-03-09|
|    17|      1003|   Debit|  4000|2024-03-09|
|    15|      1011|   Debit|  9000|2024-03-08|
+------+----------+--------+------+----------+
only showing top 5 rows


## Section 4 — Aggregations

In [32]:
# 31. Find total account balance
accounts.agg(F.sum("balance").alias("total_balance")).show()

+-------------+
|total_balance|
+-------------+
|      1055000|
+-------------+



In [33]:
# 32. Find average account balance
accounts.agg(F.avg("balance").alias("avg_balance")).show()

+-----------------+
|      avg_balance|
+-----------------+
|87916.66666666667|
+-----------------+



In [34]:
# 33. Find highest account balance
accounts.agg(F.max("balance").alias("max_balance")).show()

+-----------+
|max_balance|
+-----------+
|     200000|
+-----------+



In [35]:
# 34. Find lowest account balance
accounts.agg(F.min("balance").alias("min_balance")).show()

+-----------+
|min_balance|
+-----------+
|      30000|
+-----------+



In [36]:
# 35. Count customers by city
customers.groupBy("city").count().orderBy("count", ascending=False).show()

+---------+-----+
|     city|count|
+---------+-----+
|    Delhi|    3|
|Hyderabad|    3|
|Bangalore|    2|
|   Mumbai|    2|
|  Chennai|    1|
|     Pune|    1|
+---------+-----+



In [37]:
# 36. Count customers by account type
customers.groupBy("account_type").count().show()

+------------+-----+
|account_type|count|
+------------+-----+
|     Savings|    8|
|     Current|    4|
+------------+-----+



In [38]:
# 37. Count transactions by transaction type
transactions.groupBy("txn_type").count().show()

+--------+-----+
|txn_type|count|
+--------+-----+
|  Credit|   10|
|   Debit|   10|
+--------+-----+



In [39]:
# 38. Find total Credit transaction amount
transactions.filter(F.col("txn_type") == "Credit") \
    .agg(F.sum("amount").alias("total_credit")).show()

+------------+
|total_credit|
+------------+
|      279000|
+------------+



In [40]:
# 39. Find total Debit transaction amount
transactions.filter(F.col("txn_type") == "Debit") \
    .agg(F.sum("amount").alias("total_debit")).show()

+-----------+
|total_debit|
+-----------+
|     132000|
+-----------+



In [41]:
# 40. Find average transaction amount
transactions.agg(F.avg("amount").alias("avg_txn_amount")).show()

+--------------+
|avg_txn_amount|
+--------------+
|       20550.0|
+--------------+



## Section 5 — GroupBy Analytics

In [42]:
# 41. Find total balance by branch
accounts.groupBy("branch").agg(F.sum("balance").alias("total_balance")).orderBy("total_balance", ascending=False).show()

+-----------------+-------------+
|           branch|total_balance|
+-----------------+-------------+
|   Hyderabad Main|       285000|
|      Mumbai West|       245000|
|Bangalore Central|       230000|
|      Delhi North|       205000|
|    Chennai South|        60000|
|        Pune East|        30000|
+-----------------+-------------+



In [43]:
# 42. Find average balance by branch
accounts.groupBy("branch").agg(F.avg("balance").alias("avg_balance")).show()

+-----------------+-----------------+
|           branch|      avg_balance|
+-----------------+-----------------+
|      Delhi North|68333.33333333333|
|   Hyderabad Main|          95000.0|
|        Pune East|          30000.0|
|      Mumbai West|         122500.0|
|    Chennai South|          60000.0|
|Bangalore Central|         115000.0|
+-----------------+-----------------+



In [44]:
# 43. Count accounts by branch
accounts.groupBy("branch").count().orderBy("count", ascending=False).show()

+-----------------+-----+
|           branch|count|
+-----------------+-----+
|      Delhi North|    3|
|   Hyderabad Main|    3|
|      Mumbai West|    2|
|Bangalore Central|    2|
|        Pune East|    1|
|    Chennai South|    1|
+-----------------+-----+



In [45]:
# 44. Find total transaction amount by account_id
transactions.groupBy("account_id").agg(F.sum("amount").alias("total_amount")).orderBy("total_amount", ascending=False).show()

+----------+------------+
|account_id|total_amount|
+----------+------------+
|      1010|       95000|
|      1006|       70000|
|      1009|       54000|
|      1005|       41000|
|      1002|       33000|
|      1001|       32000|
|      1004|       27000|
|      1012|       16000|
|      1003|       14000|
|      1008|       12000|
|      1011|        9000|
|      1007|        8000|
+----------+------------+



In [46]:
# 45. Find transaction count by account_id
transactions.groupBy("account_id").count().orderBy("count", ascending=False).show()

+----------+-----+
|account_id|count|
+----------+-----+
|      1005|    2|
|      1010|    2|
|      1002|    2|
|      1001|    2|
|      1006|    2|
|      1003|    2|
|      1004|    2|
|      1009|    2|
|      1008|    1|
|      1007|    1|
|      1011|    1|
|      1012|    1|
+----------+-----+



In [47]:
# 46. Find total transaction amount by txn_type
transactions.groupBy("txn_type").agg(F.sum("amount").alias("total_amount")).show()

+--------+------------+
|txn_type|total_amount|
+--------+------------+
|  Credit|      279000|
|   Debit|      132000|
+--------+------------+



In [48]:
# 47. Find transaction count by date
transactions.groupBy("txn_date").count().orderBy("txn_date").show()

+----------+-----+
|  txn_date|count|
+----------+-----+
|2024-03-01|    2|
|2024-03-02|    2|
|2024-03-03|    2|
|2024-03-04|    2|
|2024-03-05|    2|
|2024-03-06|    2|
|2024-03-07|    2|
|2024-03-08|    2|
|2024-03-09|    2|
|2024-03-10|    2|
+----------+-----+



In [49]:
# 48. Find average customer age by city
customers.groupBy("city").agg(F.avg("age").alias("avg_age")).show()

+---------+------------------+
|     city|           avg_age|
+---------+------------------+
|Bangalore|              30.0|
|  Chennai|              30.0|
|   Mumbai|              34.5|
|     Pune|              38.0|
|    Delhi|28.666666666666668|
|Hyderabad|31.333333333333332|
+---------+------------------+



In [50]:
# 49. Find total balance by account type after joining customers and accounts
customers.join(accounts, on="customer_id", how="inner") \
    .groupBy("account_type").agg(F.sum("balance").alias("total_balance")).show()

+------------+-------------+
|account_type|total_balance|
+------------+-------------+
|     Savings|       645000|
|     Current|       410000|
+------------+-------------+



In [51]:
# 50. Find count of Savings and Current accounts by city
customers.groupBy("city", "account_type").count().orderBy("city").show()

+---------+------------+-----+
|     city|account_type|count|
+---------+------------+-----+
|Bangalore|     Savings|    1|
|Bangalore|     Current|    1|
|  Chennai|     Current|    1|
|    Delhi|     Savings|    3|
|Hyderabad|     Savings|    3|
|   Mumbai|     Savings|    1|
|   Mumbai|     Current|    1|
|     Pune|     Current|    1|
+---------+------------+-----+



## Section 6 — Joins

In [52]:
# 51. Join customers with accounts using customer_id
customers.join(accounts, on="customer_id", how="inner").show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|      1001|   Hyderabad Main|  85000|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|      1003|      Mumbai West|  45000|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad Main| 150000|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|      1007|        Pune East|  30000|
|          8|  Neha|    Delhi|

In [53]:
# 52. Show customer name, city, branch, and balance
customers.join(accounts, on="customer_id", how="inner") \
    .select("name", "city", "branch", "balance").show()

+------+---------+-----------------+-------+
|  name|     city|           branch|balance|
+------+---------+-----------------+-------+
| Rahul|Hyderabad|   Hyderabad Main|  85000|
| Sneha|Bangalore|Bangalore Central| 120000|
| Arjun|   Mumbai|      Mumbai West|  45000|
| Priya|    Delhi|      Delhi North|  95000|
| Karan|  Chennai|    Chennai South|  60000|
| Meera|Hyderabad|   Hyderabad Main| 150000|
|  Amit|     Pune|        Pune East|  30000|
|  Neha|    Delhi|      Delhi North|  70000|
| Divya|Bangalore|Bangalore Central| 110000|
|Vikram|   Mumbai|      Mumbai West| 200000|
|Farhan|Hyderabad|   Hyderabad Main|  50000|
|Simran|    Delhi|      Delhi North|  40000|
+------+---------+-----------------+-------+



In [54]:
# 53. Join accounts with transactions using account_id
accounts.join(transactions, on="account_id", how="inner").show()

+----------+-----------+-----------------+-------+------+--------+------+----------+
|account_id|customer_id|           branch|balance|txn_id|txn_type|amount|  txn_date|
+----------+-----------+-----------------+-------+------+--------+------+----------+
|      1001|          1|   Hyderabad Main|  85000|     1|  Credit| 25000|2024-03-01|
|      1002|          2|Bangalore Central| 120000|     2|   Debit| 15000|2024-03-01|
|      1003|          3|      Mumbai West|  45000|     3|  Credit| 10000|2024-03-02|
|      1004|          4|      Delhi North|  95000|     4|   Debit|  5000|2024-03-02|
|      1005|          5|    Chennai South|  60000|     5|  Credit| 30000|2024-03-03|
|      1006|          6|   Hyderabad Main| 150000|     6|   Debit| 20000|2024-03-03|
|      1007|          7|        Pune East|  30000|     7|  Credit|  8000|2024-03-04|
|      1008|          8|      Delhi North|  70000|     8|   Debit| 12000|2024-03-04|
|      1009|          9|Bangalore Central| 110000|     9|  Credit

In [55]:
# 54. Show account_id, txn_type, amount, and balance
accounts.join(transactions, on="account_id", how="inner") \
    .select("account_id", "txn_type", "amount", "balance").show()

+----------+--------+------+-------+
|account_id|txn_type|amount|balance|
+----------+--------+------+-------+
|      1001|  Credit| 25000|  85000|
|      1002|   Debit| 15000| 120000|
|      1003|  Credit| 10000|  45000|
|      1004|   Debit|  5000|  95000|
|      1005|  Credit| 30000|  60000|
|      1006|   Debit| 20000| 150000|
|      1007|  Credit|  8000|  30000|
|      1008|   Debit| 12000|  70000|
|      1009|  Credit| 40000| 110000|
|      1010|   Debit| 35000| 200000|
|      1001|   Debit|  7000|  85000|
|      1002|  Credit| 18000| 120000|
|      1006|  Credit| 50000| 150000|
|      1010|  Credit| 60000| 200000|
|      1011|   Debit|  9000|  50000|
|      1012|  Credit| 16000|  40000|
|      1003|   Debit|  4000|  45000|
|      1004|  Credit| 22000|  95000|
|      1005|   Debit| 11000|  60000|
|      1009|   Debit| 14000| 110000|
+----------+--------+------+-------+



In [56]:
# 55. Join accounts with branches using branch
accounts.join(branches, on="branch", how="inner").show()

+-----------------+----------+-----------+-------+------+-------+
|           branch|account_id|customer_id|balance|region|manager|
+-----------------+----------+-----------+-------+------+-------+
|   Hyderabad Main|      1001|          1|  85000| South| Ramesh|
|Bangalore Central|      1002|          2| 120000| South|  Leena|
|      Mumbai West|      1003|          3|  45000|  West| Joseph|
|      Delhi North|      1004|          4|  95000| North|   Sara|
|    Chennai South|      1005|          5|  60000| South|  Kumar|
|   Hyderabad Main|      1006|          6| 150000| South| Ramesh|
|        Pune East|      1007|          7|  30000|  West|  Anita|
|      Delhi North|      1008|          8|  70000| North|   Sara|
|Bangalore Central|      1009|          9| 110000| South|  Leena|
|      Mumbai West|      1010|         10| 200000|  West| Joseph|
|   Hyderabad Main|      1011|         11|  50000| South| Ramesh|
|      Delhi North|      1012|         12|  40000| North|   Sara|
+---------

In [57]:
# 56. Show branch, region, manager, and balance
accounts.join(branches, on="branch", how="inner") \
    .select("branch", "region", "manager", "balance").show()

+-----------------+------+-------+-------+
|           branch|region|manager|balance|
+-----------------+------+-------+-------+
|   Hyderabad Main| South| Ramesh|  85000|
|Bangalore Central| South|  Leena| 120000|
|      Mumbai West|  West| Joseph|  45000|
|      Delhi North| North|   Sara|  95000|
|    Chennai South| South|  Kumar|  60000|
|   Hyderabad Main| South| Ramesh| 150000|
|        Pune East|  West|  Anita|  30000|
|      Delhi North| North|   Sara|  70000|
|Bangalore Central| South|  Leena| 110000|
|      Mumbai West|  West| Joseph| 200000|
|   Hyderabad Main| South| Ramesh|  50000|
|      Delhi North| North|   Sara|  40000|
+-----------------+------+-------+-------+



In [58]:
# 57. Join customers, accounts, and transactions
customers.join(accounts, on="customer_id", how="inner") \
         .join(transactions, on="account_id", how="inner").show()

+----------+-----------+------+---------+---+------------+-----------+-----------------+-------+------+--------+------+----------+
|account_id|customer_id|  name|     city|age|account_type|signup_date|           branch|balance|txn_id|txn_type|amount|  txn_date|
+----------+-----------+------+---------+---+------------+-----------+-----------------+-------+------+--------+------+----------+
|      1001|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|   Hyderabad Main|  85000|    11|   Debit|  7000|2024-03-06|
|      1001|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|   Hyderabad Main|  85000|     1|  Credit| 25000|2024-03-01|
|      1002|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|Bangalore Central| 120000|    12|  Credit| 18000|2024-03-06|
|      1002|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|Bangalore Central| 120000|     2|   Debit| 15000|2024-03-01|
|      1003|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|      Mumbai 

In [59]:
# 58. Show customer name, city, txn_type, amount, and txn_date
customers.join(accounts, on="customer_id", how="inner") \
         .join(transactions, on="account_id", how="inner") \
         .select("name", "city", "txn_type", "amount", "txn_date").show()

+------+---------+--------+------+----------+
|  name|     city|txn_type|amount|  txn_date|
+------+---------+--------+------+----------+
| Rahul|Hyderabad|   Debit|  7000|2024-03-06|
| Rahul|Hyderabad|  Credit| 25000|2024-03-01|
| Sneha|Bangalore|  Credit| 18000|2024-03-06|
| Sneha|Bangalore|   Debit| 15000|2024-03-01|
| Arjun|   Mumbai|   Debit|  4000|2024-03-09|
| Arjun|   Mumbai|  Credit| 10000|2024-03-02|
| Priya|    Delhi|  Credit| 22000|2024-03-09|
| Priya|    Delhi|   Debit|  5000|2024-03-02|
| Karan|  Chennai|   Debit| 11000|2024-03-10|
| Karan|  Chennai|  Credit| 30000|2024-03-03|
| Meera|Hyderabad|  Credit| 50000|2024-03-07|
| Meera|Hyderabad|   Debit| 20000|2024-03-03|
|  Amit|     Pune|  Credit|  8000|2024-03-04|
|  Neha|    Delhi|   Debit| 12000|2024-03-04|
| Divya|Bangalore|   Debit| 14000|2024-03-10|
| Divya|Bangalore|  Credit| 40000|2024-03-05|
|Vikram|   Mumbai|  Credit| 60000|2024-03-07|
|Vikram|   Mumbai|   Debit| 35000|2024-03-05|
|Farhan|Hyderabad|   Debit|  9000|

In [60]:
# 59. Join customers, accounts, branches, and transactions
customers.join(accounts, on="customer_id", how="inner") \
         .join(branches, on="branch", how="inner") \
         .join(transactions, on="account_id", how="inner").show()

+----------+-----------------+-----------+------+---------+---+------------+-----------+-------+------+-------+------+--------+------+----------+
|account_id|           branch|customer_id|  name|     city|age|account_type|signup_date|balance|region|manager|txn_id|txn_type|amount|  txn_date|
+----------+-----------------+-----------+------+---------+---+------------+-----------+-------+------+-------+------+--------+------+----------+
|      1001|   Hyderabad Main|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|  85000| South| Ramesh|    11|   Debit|  7000|2024-03-06|
|      1001|   Hyderabad Main|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|  85000| South| Ramesh|     1|  Credit| 25000|2024-03-01|
|      1002|Bangalore Central|          2| Sneha|Bangalore| 32|     Current| 2023-02-12| 120000| South|  Leena|    12|  Credit| 18000|2024-03-06|
|      1002|Bangalore Central|          2| Sneha|Bangalore| 32|     Current| 2023-02-12| 120000| South|  Leena|     2|   Deb

In [61]:
# 60. Find total transaction amount by customer name
customers.join(accounts, on="customer_id", how="inner") \
         .join(transactions, on="account_id", how="inner") \
         .groupBy("name").agg(F.sum("amount").alias("total_txn_amount")) \
         .orderBy("total_txn_amount", ascending=False).show()

+------+----------------+
|  name|total_txn_amount|
+------+----------------+
|Vikram|           95000|
| Meera|           70000|
| Divya|           54000|
| Karan|           41000|
| Sneha|           33000|
| Rahul|           32000|
| Priya|           27000|
|Simran|           16000|
| Arjun|           14000|
|  Neha|           12000|
|Farhan|            9000|
|  Amit|            8000|
+------+----------------+



## Section 7 — Updating Data with withColumn

In [62]:
# 61. Add balance_in_lakhs = balance / 100000
accounts.withColumn("balance_in_lakhs", F.round(F.col("balance") / 100000, 2)).show()

+----------+-----------+-----------------+-------+----------------+
|account_id|customer_id|           branch|balance|balance_in_lakhs|
+----------+-----------+-----------------+-------+----------------+
|      1001|          1|   Hyderabad Main|  85000|            0.85|
|      1002|          2|Bangalore Central| 120000|             1.2|
|      1003|          3|      Mumbai West|  45000|            0.45|
|      1004|          4|      Delhi North|  95000|            0.95|
|      1005|          5|    Chennai South|  60000|             0.6|
|      1006|          6|   Hyderabad Main| 150000|             1.5|
|      1007|          7|        Pune East|  30000|             0.3|
|      1008|          8|      Delhi North|  70000|             0.7|
|      1009|          9|Bangalore Central| 110000|             1.1|
|      1010|         10|      Mumbai West| 200000|             2.0|
|      1011|         11|   Hyderabad Main|  50000|             0.5|
|      1012|         12|      Delhi North|  4000

In [63]:
# 62. Add bank_name = "BotCampus Bank"
customers.withColumn("bank_name", F.lit("BotCampus Bank")).show()

+-----------+------+---------+---+------------+-----------+--------------+
|customer_id|  name|     city|age|account_type|signup_date|     bank_name|
+-----------+------+---------+---+------------+-----------+--------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|BotCampus Bank|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|BotCampus Bank|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|BotCampus Bank|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|BotCampus Bank|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|BotCampus Bank|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|BotCampus Bank|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|BotCampus Bank|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|BotCampus Bank|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|BotCampus Bank|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|BotCampus Bank|
|         11|Farhan|Hyder

In [64]:
# 63. Add annual_service_fee = balance * 0.01
accounts.withColumn("annual_service_fee", F.round(F.col("balance") * 0.01, 2)).show()

+----------+-----------+-----------------+-------+------------------+
|account_id|customer_id|           branch|balance|annual_service_fee|
+----------+-----------+-----------------+-------+------------------+
|      1001|          1|   Hyderabad Main|  85000|             850.0|
|      1002|          2|Bangalore Central| 120000|            1200.0|
|      1003|          3|      Mumbai West|  45000|             450.0|
|      1004|          4|      Delhi North|  95000|             950.0|
|      1005|          5|    Chennai South|  60000|             600.0|
|      1006|          6|   Hyderabad Main| 150000|            1500.0|
|      1007|          7|        Pune East|  30000|             300.0|
|      1008|          8|      Delhi North|  70000|             700.0|
|      1009|          9|Bangalore Central| 110000|            1100.0|
|      1010|         10|      Mumbai West| 200000|            2000.0|
|      1011|         11|   Hyderabad Main|  50000|             500.0|
|      1012|        

In [65]:
# 64. Add net_balance = balance - annual_service_fee
accounts.withColumn("annual_service_fee", F.round(F.col("balance") * 0.01, 2)) \
        .withColumn("net_balance", F.col("balance") - F.round(F.col("balance") * 0.01, 2)).show()

+----------+-----------+-----------------+-------+------------------+-----------+
|account_id|customer_id|           branch|balance|annual_service_fee|net_balance|
+----------+-----------+-----------------+-------+------------------+-----------+
|      1001|          1|   Hyderabad Main|  85000|             850.0|    84150.0|
|      1002|          2|Bangalore Central| 120000|            1200.0|   118800.0|
|      1003|          3|      Mumbai West|  45000|             450.0|    44550.0|
|      1004|          4|      Delhi North|  95000|             950.0|    94050.0|
|      1005|          5|    Chennai South|  60000|             600.0|    59400.0|
|      1006|          6|   Hyderabad Main| 150000|            1500.0|   148500.0|
|      1007|          7|        Pune East|  30000|             300.0|    29700.0|
|      1008|          8|      Delhi North|  70000|             700.0|    69300.0|
|      1009|          9|Bangalore Central| 110000|            1100.0|   108900.0|
|      1010|    

In [66]:
# 65. Add is_high_balance where balance > 100000
accounts.withColumn("is_high_balance",
    F.when(F.col("balance") > 100000, True).otherwise(False)
).show()

+----------+-----------+-----------------+-------+---------------+
|account_id|customer_id|           branch|balance|is_high_balance|
+----------+-----------+-----------------+-------+---------------+
|      1001|          1|   Hyderabad Main|  85000|          false|
|      1002|          2|Bangalore Central| 120000|           true|
|      1003|          3|      Mumbai West|  45000|          false|
|      1004|          4|      Delhi North|  95000|          false|
|      1005|          5|    Chennai South|  60000|          false|
|      1006|          6|   Hyderabad Main| 150000|           true|
|      1007|          7|        Pune East|  30000|          false|
|      1008|          8|      Delhi North|  70000|          false|
|      1009|          9|Bangalore Central| 110000|           true|
|      1010|         10|      Mumbai West| 200000|           true|
|      1011|         11|   Hyderabad Main|  50000|          false|
|      1012|         12|      Delhi North|  40000|          fa

In [67]:
# 66. Add txn_amount_in_k = amount / 1000
transactions.withColumn("txn_amount_in_k", F.round(F.col("amount") / 1000, 2)).show()

+------+----------+--------+------+----------+---------------+
|txn_id|account_id|txn_type|amount|  txn_date|txn_amount_in_k|
+------+----------+--------+------+----------+---------------+
|     1|      1001|  Credit| 25000|2024-03-01|           25.0|
|     2|      1002|   Debit| 15000|2024-03-01|           15.0|
|     3|      1003|  Credit| 10000|2024-03-02|           10.0|
|     4|      1004|   Debit|  5000|2024-03-02|            5.0|
|     5|      1005|  Credit| 30000|2024-03-03|           30.0|
|     6|      1006|   Debit| 20000|2024-03-03|           20.0|
|     7|      1007|  Credit|  8000|2024-03-04|            8.0|
|     8|      1008|   Debit| 12000|2024-03-04|           12.0|
|     9|      1009|  Credit| 40000|2024-03-05|           40.0|
|    10|      1010|   Debit| 35000|2024-03-05|           35.0|
|    11|      1001|   Debit|  7000|2024-03-06|            7.0|
|    12|      1002|  Credit| 18000|2024-03-06|           18.0|
|    13|      1006|  Credit| 50000|2024-03-07|         

In [68]:
# 67. Add country = "India" to customers
customers.withColumn("country", F.lit("India")).show()

+-----------+------+---------+---+------------+-----------+-------+
|customer_id|  name|     city|age|account_type|signup_date|country|
+-----------+------+---------+---+------------+-----------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|  India|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|  India|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|  India|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|  India|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|  India|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|  India|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|  India|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|  India|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|  India|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|  India|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|  India|
|         12|Simran|    Delhi| 25|     Savings| 

In [69]:
# 68. Add customer_label = name + " - " + city
customers.withColumn("customer_label",
    F.concat(F.col("name"), F.lit(" - "), F.col("city"))
).show()

+-----------+------+---------+---+------------+-----------+------------------+
|customer_id|  name|     city|age|account_type|signup_date|    customer_label|
+-----------+------+---------+---+------------+-----------+------------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10| Rahul - Hyderabad|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12| Sneha - Bangalore|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|    Arjun - Mumbai|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|     Priya - Delhi|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|   Karan - Chennai|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10| Meera - Hyderabad|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|       Amit - Pune|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|      Neha - Delhi|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15| Divya - Bangalore|
|         10|Vikram|   Mumbai| 42|     Current| 2023

In [70]:
# 69. Add branch_label = branch + " - " + region
accounts.join(branches, on="branch", how="inner") \
    .withColumn("branch_label",
        F.concat(F.col("branch"), F.lit(" - "), F.col("region"))
    ).select("account_id", "branch", "region", "branch_label").show()

+----------+-----------------+------+--------------------+
|account_id|           branch|region|        branch_label|
+----------+-----------------+------+--------------------+
|      1001|   Hyderabad Main| South|Hyderabad Main - ...|
|      1002|Bangalore Central| South|Bangalore Central...|
|      1003|      Mumbai West|  West|  Mumbai West - West|
|      1004|      Delhi North| North| Delhi North - North|
|      1005|    Chennai South| South|Chennai South - S...|
|      1006|   Hyderabad Main| South|Hyderabad Main - ...|
|      1007|        Pune East|  West|    Pune East - West|
|      1008|      Delhi North| North| Delhi North - North|
|      1009|Bangalore Central| South|Bangalore Central...|
|      1010|      Mumbai West|  West|  Mumbai West - West|
|      1011|   Hyderabad Main| South|Hyderabad Main - ...|
|      1012|      Delhi North| North| Delhi North - North|
+----------+-----------------+------+--------------------+



In [71]:
# 70. Add risk_flag for transactions above 40000
transactions.withColumn("risk_flag",
    F.when(F.col("amount") > 40000, True).otherwise(False)
).show()

+------+----------+--------+------+----------+---------+
|txn_id|account_id|txn_type|amount|  txn_date|risk_flag|
+------+----------+--------+------+----------+---------+
|     1|      1001|  Credit| 25000|2024-03-01|    false|
|     2|      1002|   Debit| 15000|2024-03-01|    false|
|     3|      1003|  Credit| 10000|2024-03-02|    false|
|     4|      1004|   Debit|  5000|2024-03-02|    false|
|     5|      1005|  Credit| 30000|2024-03-03|    false|
|     6|      1006|   Debit| 20000|2024-03-03|    false|
|     7|      1007|  Credit|  8000|2024-03-04|    false|
|     8|      1008|   Debit| 12000|2024-03-04|    false|
|     9|      1009|  Credit| 40000|2024-03-05|    false|
|    10|      1010|   Debit| 35000|2024-03-05|    false|
|    11|      1001|   Debit|  7000|2024-03-06|    false|
|    12|      1002|  Credit| 18000|2024-03-06|    false|
|    13|      1006|  Credit| 50000|2024-03-07|     true|
|    14|      1010|  Credit| 60000|2024-03-07|     true|
|    15|      1011|   Debit|  9

:## Section 8 — When / Otherwise

In [72]:
# 71. Create balance category: High if balance >= 100000, Medium if >= 50000, else Low
accounts.withColumn("balance_category",
    F.when(F.col("balance") >= 100000, "High")
     .when(F.col("balance") >= 50000, "Medium")
     .otherwise("Low")
).select("account_id", "balance", "balance_category").show()

+----------+-------+----------------+
|account_id|balance|balance_category|
+----------+-------+----------------+
|      1001|  85000|          Medium|
|      1002| 120000|            High|
|      1003|  45000|             Low|
|      1004|  95000|          Medium|
|      1005|  60000|          Medium|
|      1006| 150000|            High|
|      1007|  30000|             Low|
|      1008|  70000|          Medium|
|      1009| 110000|            High|
|      1010| 200000|            High|
|      1011|  50000|          Medium|
|      1012|  40000|             Low|
+----------+-------+----------------+



In [73]:
# 72. Create age group: Young if age < 30, Adult if age < 40, else Senior
customers.withColumn("age_group",
    F.when(F.col("age") < 30, "Young")
     .when(F.col("age") < 40, "Adult")
     .otherwise("Senior")
).select("name", "age", "age_group").show()

+------+---+---------+
|  name|age|age_group|
+------+---+---------+
| Rahul| 29|    Young|
| Sneha| 32|    Adult|
| Arjun| 27|    Young|
| Priya| 35|    Adult|
| Karan| 30|    Adult|
| Meera| 31|    Adult|
|  Amit| 38|    Adult|
|  Neha| 26|    Young|
| Divya| 28|    Young|
|Vikram| 42|   Senior|
|Farhan| 34|    Adult|
|Simran| 25|    Young|
+------+---+---------+



In [74]:
# 73. Create transaction category: Large if amount >= 30000, Medium if >= 10000, else Small
transactions.withColumn("txn_category",
    F.when(F.col("amount") >= 30000, "Large")
     .when(F.col("amount") >= 10000, "Medium")
     .otherwise("Small")
).select("txn_id", "amount", "txn_category").show()

+------+------+------------+
|txn_id|amount|txn_category|
+------+------+------------+
|     1| 25000|      Medium|
|     2| 15000|      Medium|
|     3| 10000|      Medium|
|     4|  5000|       Small|
|     5| 30000|       Large|
|     6| 20000|      Medium|
|     7|  8000|       Small|
|     8| 12000|      Medium|
|     9| 40000|       Large|
|    10| 35000|       Large|
|    11|  7000|       Small|
|    12| 18000|      Medium|
|    13| 50000|       Large|
|    14| 60000|       Large|
|    15|  9000|       Small|
|    16| 16000|      Medium|
|    17|  4000|       Small|
|    18| 22000|      Medium|
|    19| 11000|      Medium|
|    20| 14000|      Medium|
+------+------+------------+



In [75]:
# 74. Create region priority: South = High Priority, North = Medium Priority, West = Watch
branches.withColumn("region_priority",
    F.when(F.col("region") == "South", "High Priority")
     .when(F.col("region") == "North", "Medium Priority")
     .otherwise("Watch")
).show()

+-----------------+------+-------+---------------+
|           branch|region|manager|region_priority|
+-----------------+------+-------+---------------+
|   Hyderabad Main| South| Ramesh|  High Priority|
|Bangalore Central| South|  Leena|  High Priority|
|      Mumbai West|  West| Joseph|          Watch|
|      Delhi North| North|   Sara|Medium Priority|
|    Chennai South| South|  Kumar|  High Priority|
|        Pune East|  West|  Anita|          Watch|
+-----------------+------+-------+---------------+



In [76]:
# 75. Create account type label: Savings = Retail, Current = Business
customers.withColumn("account_label",
    F.when(F.col("account_type") == "Savings", "Retail")
     .otherwise("Business")
).select("name", "account_type", "account_label").show()

+------+------------+-------------+
|  name|account_type|account_label|
+------+------------+-------------+
| Rahul|     Savings|       Retail|
| Sneha|     Current|     Business|
| Arjun|     Savings|       Retail|
| Priya|     Savings|       Retail|
| Karan|     Current|     Business|
| Meera|     Savings|       Retail|
|  Amit|     Current|     Business|
|  Neha|     Savings|       Retail|
| Divya|     Savings|       Retail|
|Vikram|     Current|     Business|
|Farhan|     Savings|       Retail|
|Simran|     Savings|       Retail|
+------+------------+-------------+



## Section 9 — Date Functions

In [77]:
# 76. Convert signup_date to date type
customers.withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd")).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [78]:
# 77. Extract signup year
customers.withColumn("signup_year", F.year(F.to_date("signup_date"))) \
    .select("name", "signup_date", "signup_year").show()

+------+-----------+-----------+
|  name|signup_date|signup_year|
+------+-----------+-----------+
| Rahul| 2023-01-10|       2023|
| Sneha| 2023-02-12|       2023|
| Arjun| 2023-03-14|       2023|
| Priya| 2023-04-15|       2023|
| Karan| 2023-05-11|       2023|
| Meera| 2023-06-10|       2023|
|  Amit| 2023-06-22|       2023|
|  Neha| 2023-07-10|       2023|
| Divya| 2023-07-15|       2023|
|Vikram| 2023-08-01|       2023|
|Farhan| 2023-08-10|       2023|
|Simran| 2023-08-21|       2023|
+------+-----------+-----------+



In [79]:
# 78. Extract signup month
customers.withColumn("signup_month", F.month(F.to_date("signup_date"))) \
    .select("name", "signup_date", "signup_month").show()

+------+-----------+------------+
|  name|signup_date|signup_month|
+------+-----------+------------+
| Rahul| 2023-01-10|           1|
| Sneha| 2023-02-12|           2|
| Arjun| 2023-03-14|           3|
| Priya| 2023-04-15|           4|
| Karan| 2023-05-11|           5|
| Meera| 2023-06-10|           6|
|  Amit| 2023-06-22|           6|
|  Neha| 2023-07-10|           7|
| Divya| 2023-07-15|           7|
|Vikram| 2023-08-01|           8|
|Farhan| 2023-08-10|           8|
|Simran| 2023-08-21|           8|
+------+-----------+------------+



In [80]:
# 79. Convert txn_date to date type
transactions.withColumn("txn_date", F.to_date(F.col("txn_date"), "yyyy-MM-dd")).show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    11|      1001|   Debit|  7000|2024-03-06|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    15|      1011|   Debit|  9000|2024-03-08|
|    16|      1012|  Credit| 16000|2024-03-08|
|    17|      1003|   Debit|  4000|2024-03-09|
|    18|      1004|  Credit| 22000|2024-03-09|
|    19|     

In [81]:
# 80. Extract transaction month
transactions.withColumn("txn_month", F.month(F.to_date("txn_date"))) \
    .select("txn_id", "txn_date", "txn_month").show()

+------+----------+---------+
|txn_id|  txn_date|txn_month|
+------+----------+---------+
|     1|2024-03-01|        3|
|     2|2024-03-01|        3|
|     3|2024-03-02|        3|
|     4|2024-03-02|        3|
|     5|2024-03-03|        3|
|     6|2024-03-03|        3|
|     7|2024-03-04|        3|
|     8|2024-03-04|        3|
|     9|2024-03-05|        3|
|    10|2024-03-05|        3|
|    11|2024-03-06|        3|
|    12|2024-03-06|        3|
|    13|2024-03-07|        3|
|    14|2024-03-07|        3|
|    15|2024-03-08|        3|
|    16|2024-03-08|        3|
|    17|2024-03-09|        3|
|    18|2024-03-09|        3|
|    19|2024-03-10|        3|
|    20|2024-03-10|        3|
+------+----------+---------+



In [82]:
# 81. Count transactions by transaction date
transactions.groupBy("txn_date").count().orderBy("txn_date").show()

+----------+-----+
|  txn_date|count|
+----------+-----+
|2024-03-01|    2|
|2024-03-02|    2|
|2024-03-03|    2|
|2024-03-04|    2|
|2024-03-05|    2|
|2024-03-06|    2|
|2024-03-07|    2|
|2024-03-08|    2|
|2024-03-09|    2|
|2024-03-10|    2|
+----------+-----+



In [83]:
# 82. Count transactions by month
transactions.withColumn("txn_month", F.month(F.to_date("txn_date"))) \
    .groupBy("txn_month").count().orderBy("txn_month").show()

+---------+-----+
|txn_month|count|
+---------+-----+
|        3|   20|
+---------+-----+



In [84]:
# 83. Find customers signed up after 2023-06-01
customers.filter(F.to_date("signup_date") > F.lit("2023-06-01")).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [85]:
# 84. Find days since signup
customers.withColumn("days_since_signup",
    F.datediff(F.current_date(), F.to_date("signup_date"))
).select("name", "signup_date", "days_since_signup").show()

+------+-----------+-----------------+
|  name|signup_date|days_since_signup|
+------+-----------+-----------------+
| Rahul| 2023-01-10|             1203|
| Sneha| 2023-02-12|             1170|
| Arjun| 2023-03-14|             1140|
| Priya| 2023-04-15|             1108|
| Karan| 2023-05-11|             1082|
| Meera| 2023-06-10|             1052|
|  Amit| 2023-06-22|             1040|
|  Neha| 2023-07-10|             1022|
| Divya| 2023-07-15|             1017|
|Vikram| 2023-08-01|             1000|
|Farhan| 2023-08-10|              991|
|Simran| 2023-08-21|              980|
+------+-----------+-----------------+



In [86]:
# 85. Find days since transaction
transactions.withColumn("days_since_txn",
    F.datediff(F.current_date(), F.to_date("txn_date"))
).select("txn_id", "txn_date", "days_since_txn").show()

+------+----------+--------------+
|txn_id|  txn_date|days_since_txn|
+------+----------+--------------+
|     1|2024-03-01|           787|
|     2|2024-03-01|           787|
|     3|2024-03-02|           786|
|     4|2024-03-02|           786|
|     5|2024-03-03|           785|
|     6|2024-03-03|           785|
|     7|2024-03-04|           784|
|     8|2024-03-04|           784|
|     9|2024-03-05|           783|
|    10|2024-03-05|           783|
|    11|2024-03-06|           782|
|    12|2024-03-06|           782|
|    13|2024-03-07|           781|
|    14|2024-03-07|           781|
|    15|2024-03-08|           780|
|    16|2024-03-08|           780|
|    17|2024-03-09|           779|
|    18|2024-03-09|           779|
|    19|2024-03-10|           778|
|    20|2024-03-10|           778|
+------+----------+--------------+



## Section 10 — Window Functions

In [87]:
# 86. Rank customers by balance within city
win = Window.partitionBy("city").orderBy(F.col("balance").desc())
customers.join(accounts, on="customer_id", how="inner") \
    .withColumn("balance_rank_in_city", F.rank().over(win)) \
    .select("name", "city", "balance", "balance_rank_in_city").show()

+------+---------+-------+--------------------+
|  name|     city|balance|balance_rank_in_city|
+------+---------+-------+--------------------+
| Sneha|Bangalore| 120000|                   1|
| Divya|Bangalore| 110000|                   2|
| Karan|  Chennai|  60000|                   1|
| Priya|    Delhi|  95000|                   1|
|  Neha|    Delhi|  70000|                   2|
|Simran|    Delhi|  40000|                   3|
| Meera|Hyderabad| 150000|                   1|
| Rahul|Hyderabad|  85000|                   2|
|Farhan|Hyderabad|  50000|                   3|
|Vikram|   Mumbai| 200000|                   1|
| Arjun|   Mumbai|  45000|                   2|
|  Amit|     Pune|  30000|                   1|
+------+---------+-------+--------------------+



In [88]:
# 87. Use row_number to find top balance account per city
win = Window.partitionBy("city").orderBy(F.col("balance").desc())
customers.join(accounts, on="customer_id", how="inner") \
    .withColumn("rn", F.row_number().over(win)) \
    .filter(F.col("rn") == 1) \
    .select("name", "city", "balance").show()

+------+---------+-------+
|  name|     city|balance|
+------+---------+-------+
| Sneha|Bangalore| 120000|
| Karan|  Chennai|  60000|
| Priya|    Delhi|  95000|
| Meera|Hyderabad| 150000|
|Vikram|   Mumbai| 200000|
|  Amit|     Pune|  30000|
+------+---------+-------+



In [89]:
# 88. Use dense_rank to rank transaction amount by transaction type
win = Window.partitionBy("txn_type").orderBy(F.col("amount").desc())
transactions.withColumn("amount_dense_rank", F.dense_rank().over(win)).show()

+------+----------+--------+------+----------+-----------------+
|txn_id|account_id|txn_type|amount|  txn_date|amount_dense_rank|
+------+----------+--------+------+----------+-----------------+
|    14|      1010|  Credit| 60000|2024-03-07|                1|
|    13|      1006|  Credit| 50000|2024-03-07|                2|
|     9|      1009|  Credit| 40000|2024-03-05|                3|
|     5|      1005|  Credit| 30000|2024-03-03|                4|
|     1|      1001|  Credit| 25000|2024-03-01|                5|
|    18|      1004|  Credit| 22000|2024-03-09|                6|
|    12|      1002|  Credit| 18000|2024-03-06|                7|
|    16|      1012|  Credit| 16000|2024-03-08|                8|
|     3|      1003|  Credit| 10000|2024-03-02|                9|
|     7|      1007|  Credit|  8000|2024-03-04|               10|
|    10|      1010|   Debit| 35000|2024-03-05|                1|
|     6|      1006|   Debit| 20000|2024-03-03|                2|
|     2|      1002|   Deb

In [90]:
# 89. Find top 2 transactions per account
win = Window.partitionBy("account_id").orderBy(F.col("amount").desc())
transactions.withColumn("rn", F.row_number().over(win)) \
    .filter(F.col("rn") <= 2).drop("rn").show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|    11|      1001|   Debit|  7000|2024-03-06|
|    12|      1002|  Credit| 18000|2024-03-06|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|    17|      1003|   Debit|  4000|2024-03-09|
|    18|      1004|  Credit| 22000|2024-03-09|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|    19|      1005|   Debit| 11000|2024-03-10|
|    13|      1006|  Credit| 50000|2024-03-07|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    20|      1009|   Debit| 14000|2024-03-10|
|    14|      1010|  Credit| 60000|2024-03-07|
|    10|      1010|   Debit| 35000|2024-03-05|
|    15|     

In [91]:
# 90. Create running transaction total per account_id
win = Window.partitionBy("account_id").orderBy("txn_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
transactions.withColumn("running_total", F.sum("amount").over(win)).show()

+------+----------+--------+------+----------+-------------+
|txn_id|account_id|txn_type|amount|  txn_date|running_total|
+------+----------+--------+------+----------+-------------+
|     1|      1001|  Credit| 25000|2024-03-01|        25000|
|    11|      1001|   Debit|  7000|2024-03-06|        32000|
|     2|      1002|   Debit| 15000|2024-03-01|        15000|
|    12|      1002|  Credit| 18000|2024-03-06|        33000|
|     3|      1003|  Credit| 10000|2024-03-02|        10000|
|    17|      1003|   Debit|  4000|2024-03-09|        14000|
|     4|      1004|   Debit|  5000|2024-03-02|         5000|
|    18|      1004|  Credit| 22000|2024-03-09|        27000|
|     5|      1005|  Credit| 30000|2024-03-03|        30000|
|    19|      1005|   Debit| 11000|2024-03-10|        41000|
|     6|      1006|   Debit| 20000|2024-03-03|        20000|
|    13|      1006|  Credit| 50000|2024-03-07|        70000|
|     7|      1007|  Credit|  8000|2024-03-04|         8000|
|     8|      1008|   De

In [92]:
# 91. Rank branches by total balance
branch_balance = accounts.groupBy("branch").agg(F.sum("balance").alias("total_balance"))
win = Window.orderBy(F.col("total_balance").desc())
branch_balance.withColumn("branch_rank", F.rank().over(win)).show()

+-----------------+-------------+-----------+
|           branch|total_balance|branch_rank|
+-----------------+-------------+-----------+
|   Hyderabad Main|       285000|          1|
|      Mumbai West|       245000|          2|
|Bangalore Central|       230000|          3|
|      Delhi North|       205000|          4|
|    Chennai South|        60000|          5|
|        Pune East|        30000|          6|
+-----------------+-------------+-----------+



In [93]:
# 92. Rank cities by total customer balance
city_balance = customers.join(accounts, on="customer_id", how="inner") \
    .groupBy("city").agg(F.sum("balance").alias("total_balance"))
win = Window.orderBy(F.col("total_balance").desc())
city_balance.withColumn("city_rank", F.rank().over(win)).show()

+---------+-------------+---------+
|     city|total_balance|city_rank|
+---------+-------------+---------+
|Hyderabad|       285000|        1|
|   Mumbai|       245000|        2|
|Bangalore|       230000|        3|
|    Delhi|       205000|        4|
|  Chennai|        60000|        5|
|     Pune|        30000|        6|
+---------+-------------+---------+



In [94]:
# 93. Rank customers by total transaction amount
cust_txn = customers.join(accounts, on="customer_id", how="inner") \
    .join(transactions, on="account_id", how="inner") \
    .groupBy("name").agg(F.sum("amount").alias("total_txn"))
win = Window.orderBy(F.col("total_txn").desc())
cust_txn.withColumn("txn_rank", F.rank().over(win)).show()

+------+---------+--------+
|  name|total_txn|txn_rank|
+------+---------+--------+
|Vikram|    95000|       1|
| Meera|    70000|       2|
| Divya|    54000|       3|
| Karan|    41000|       4|
| Sneha|    33000|       5|
| Rahul|    32000|       6|
| Priya|    27000|       7|
|Simran|    16000|       8|
| Arjun|    14000|       9|
|  Neha|    12000|      10|
|Farhan|     9000|      11|
|  Amit|     8000|      12|
+------+---------+--------+



In [95]:
# 94. Find highest transaction customer per city
cust_txn_city = customers.join(accounts, on="customer_id", how="inner") \
    .join(transactions, on="account_id", how="inner") \
    .groupBy("city", "name").agg(F.sum("amount").alias("total_txn"))
win = Window.partitionBy("city").orderBy(F.col("total_txn").desc())
cust_txn_city.withColumn("rn", F.row_number().over(win)) \
    .filter(F.col("rn") == 1).drop("rn").show()

+---------+------+---------+
|     city|  name|total_txn|
+---------+------+---------+
|Bangalore| Divya|    54000|
|  Chennai| Karan|    41000|
|    Delhi| Priya|    27000|
|Hyderabad| Meera|    70000|
|   Mumbai|Vikram|    95000|
|     Pune|  Amit|     8000|
+---------+------+---------+



In [96]:
# 95. Find highest balance customer per region
cust_region = customers.join(accounts, on="customer_id", how="inner") \
    .join(branches, on="branch", how="inner")
win = Window.partitionBy("region").orderBy(F.col("balance").desc())
cust_region.withColumn("rn", F.row_number().over(win)) \
    .filter(F.col("rn") == 1) \
    .select("region", "name", "balance").show()

+------+------+-------+
|region|  name|balance|
+------+------+-------+
| North| Priya|  95000|
| South| Meera| 150000|
|  West|Vikram| 200000|
+------+------+-------+



## Section 11 — Complex JSON

In [97]:
# 96. Read customer_profiles.json
profiles = spark.read.json("customer_profiles.json", multiLine=True)
profiles.show()

+--------------------+-----------+------+--------------------+
|             contact|customer_id|  name|            services|
+--------------------+-----------+------+--------------------+
|{rahul@mail.com, ...|          1| Rahul|[UPI, Credit Card...|
|{sneha@mail.com, ...|          2| Sneha|   [UPI, Debit Card]|
|{arjun@mail.com, ...|          3| Arjun| [Net Banking, Loan]|
|{meera@mail.com, ...|          6| Meera|[UPI, Credit Card...|
|{vikram@mail.com,...|         10|Vikram|[Net Banking, Wea...|
+--------------------+-----------+------+--------------------+



In [98]:
# 97. Extract customer_id, name, email, and phone
profiles.select(
    "customer_id",
    "name",
    F.col("contact.email").alias("email"),
    F.col("contact.phone").alias("phone")
).show()

+-----------+------+---------------+----------+
|customer_id|  name|          email|     phone|
+-----------+------+---------------+----------+
|          1| Rahul| rahul@mail.com|9000011111|
|          2| Sneha| sneha@mail.com|9000022222|
|          3| Arjun| arjun@mail.com|9000033333|
|          6| Meera| meera@mail.com|9000066666|
|         10|Vikram|vikram@mail.com|9000101010|
+-----------+------+---------------+----------+



In [99]:
# 98. Explode services
profiles.select("customer_id", "name", F.explode("services").alias("service")).show()

+-----------+------+-----------+
|customer_id|  name|    service|
+-----------+------+-----------+
|          1| Rahul|        UPI|
|          1| Rahul|Credit Card|
|          1| Rahul|Net Banking|
|          2| Sneha|        UPI|
|          2| Sneha| Debit Card|
|          3| Arjun|Net Banking|
|          3| Arjun|       Loan|
|          6| Meera|        UPI|
|          6| Meera|Credit Card|
|          6| Meera|       Loan|
|         10|Vikram|Net Banking|
|         10|Vikram|     Wealth|
+-----------+------+-----------+



In [100]:
# 99. Find customers using UPI
profiles.select("customer_id", "name", F.explode("services").alias("service")) \
    .filter(F.col("service") == "UPI").show()

+-----------+-----+-------+
|customer_id| name|service|
+-----------+-----+-------+
|          1|Rahul|    UPI|
|          2|Sneha|    UPI|
|          6|Meera|    UPI|
+-----------+-----+-------+



In [101]:
# 100. Count how many customers use each service
profiles.select("customer_id", F.explode("services").alias("service")) \
    .groupBy("service").count().orderBy("count", ascending=False).show()

+-----------+-----+
|    service|count|
+-----------+-----+
|Net Banking|    3|
|        UPI|    3|
|       Loan|    2|
|Credit Card|    2|
|     Wealth|    1|
| Debit Card|    1|
+-----------+-----+

